In [1]:
from transformers import pipeline

# Data handling
import pandas as pd

c:\Users\nikhi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
feedbacks = [
        "The delivery was late and the box was damaged",
        "Customer support did not respond for three days",
        "Amazing app experience, very intuitive UI",
        "Payment failed twice and money was deducted",
        "The product quality is excellent but delivery was slow",
        "App crashes frequently after the latest update",
        "Refund process was smooth and quick",
        "Terrible customer service, very disappointing",
        "Great value for money, highly recommend",
        "Login OTP never arrives, stuck outside my account",
        "Customer support is very bad",
        "Very Good Product"
]

In [9]:
# Pipeline = (Task + Model + Pre/Post Processing) 

# Model: distilbert-base-uncased-finetuned-sst-2-english
#
# Architecture:
# - DistilBERT → a lightweight, compressed version of BERT (encoder-only Transformer)
# - Uses fewer layers than BERT (6 vs 12), ~40% fewer parameters, faster inference
#
# Pretraining:
# - Self-supervised pretraining using Masked Language Modeling (MLM)
# - Bidirectional context (looks at both left and right words)
# - "uncased" → text is lowercased (case-insensitive)
#
# Fine-tuning:
# - Fine-tuned on the SST-2 (Stanford Sentiment Treebank) dataset
# - Task: binary sentiment classification (POSITIVE / NEGATIVE)
# - Classification head added on top of the [CLS] token representation
#
# Output:
# - Produces sentiment label + confidence score
# - Example labels: POSITIVE, NEGATIVE


sentiment_pipe = pipeline(
"sentiment-analysis", 
model="distilbert-base-uncased-finetuned-sst-2-english" # Encoder Model
)

sentiment_results = sentiment_pipe(feedbacks)
sentiment_results

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 201.51it/s, Materializing param=pre_classifier.weight]                                  


[{'label': 'NEGATIVE', 'score': 0.999708354473114},
 {'label': 'NEGATIVE', 'score': 0.9964256882667542},
 {'label': 'POSITIVE', 'score': 0.9998181462287903},
 {'label': 'NEGATIVE', 'score': 0.9993096590042114},
 {'label': 'NEGATIVE', 'score': 0.9938415288925171},
 {'label': 'NEGATIVE', 'score': 0.9996180534362793},
 {'label': 'POSITIVE', 'score': 0.9964451193809509},
 {'label': 'NEGATIVE', 'score': 0.9998053908348083},
 {'label': 'POSITIVE', 'score': 0.9998574256896973},
 {'label': 'NEGATIVE', 'score': 0.9997727274894714},
 {'label': 'NEGATIVE', 'score': 0.9998050332069397},
 {'label': 'POSITIVE', 'score': 0.9998735189437866}]

In [11]:
df = pd.DataFrame({
"feedback": feedbacks,
"sentiment": [s['label'] for s in sentiment_results],
"confidence": [round(s['score'], 3) for s in sentiment_results]
})


df

,feedback,sentiment,confidence
0,The delivery was late and the box was damaged,NEGATIVE,1.000
1,Customer support did not respond for three days,NEGATIVE,0.996
2,"Amazing app experience, very intuitive UI",POSITIVE,1.000
3,Payment failed twice and money was deducted,NEGATIVE,0.999
4,The product quality is excellent but delivery ...,NEGATIVE,0.994
5,App crashes frequently after the latest update,NEGATIVE,1.000
6,Refund process was smooth and quick,POSITIVE,0.996
7,"Terrible customer service, very disappointing",NEGATIVE,1.000
8,"Great value for money, highly recommend",POSITIVE,1.000
9,"Login OTP never arrives, stuck outside my account",NEGATIVE,1.000


In [15]:
# Issue Extraction using NER

# Pipeline: Named Entity Recognition (NER)
#
# Task:
# - Token-level classification to identify and label entities in text
# - Common entity types: PERSON, ORGANIZATION, LOCATION, DATE, MONEY, etc.
#
# Model (default):
# - Encoder-only Transformer (typically BERT / DistilBERT family)
# - Pretrained using Masked Language Modeling (MLM)
# - Fine-tuned on NER datasets (e.g., CoNLL-style entity tagging)
#
# How it works:
# - Input text is tokenized (often into subwords)
# - Each token receives an entity label (BIO tagging scheme)
# - Example labels: B-PER, I-PER, B-ORG, O
#
# aggregation_strategy="simple":
# - Groups subword tokens into full entities
# - Merges B- and I- tags into a single entity span
# - Produces human-readable outputs instead of token-level fragments
#
# Output:
# - Entity text span
# - Entity label (e.g., PER, ORG, LOC)
# - Confidence score
#
# Use cases:
# - Resume parsing
# - Information extraction from documents
# - Medical / legal text analysis
# - Knowledge graph construction
#
# Key idea:
# - Uses contextual token embeddings from an encoder model
# - No text generation, only structured information extraction


ner_pipe = pipeline(
"ner",
aggregation_strategy="simple"
)

for text in feedbacks:
    print(f"Feedback: {text}")
    print("Entities:", ner_pipe(text))
    print("-" * 60)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:01<00:00, 199.88it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Feedback: The delivery was late and the box was damaged
Entities: []
------------------------------------------------------------
Feedback: Customer support did not respond for three days
Entities: []
------------------------------------------------------------
Feedback: Amazing app experience, very intuitive UI
Entities: []
------------------------------------------------------------
Feedback: Payment failed twice and money was deducted
Entities: []
------------------------------------------------------------
Feedback: The product quality is excellent but delivery was slow
Entities: []
------------------------------------------------------------
Feedback: App crashes frequently after the latest update
Entities: []
------------------------------------------------------------
Feedback: Refund process was smooth and quick
Entities: []
------------------------------------------------------------
Feedback: Terrible customer service, very disappointing
Entities: []
-------------------------

In [9]:
# Negative Feedback

negative_feedbacks = df[df.sentiment == "NEGATIVE"]["feedback"].tolist()
negative_feedbacks

['The delivery was late and the box was damaged',
 'Customer support did not respond for three days',
 'Payment failed twice and money was deducted',
 'The product quality is excellent but delivery was slow',
 'App crashes frequently after the latest update',
 'Terrible customer service, very disappointing',
 'Login OTP never arrives, stuck outside my account']

In [16]:
# Zero-Shot Classification

# Pipeline: Zero-Shot Text Classification
#
# Task:
# - Classify text into user-defined labels WITHOUT task-specific training
# - Uses Natural Language Inference (NLI) instead of a traditional classifier
#
# Model:
# - facebook/bart-large-mnli
# - Architecture: BART (Encoder–Decoder Transformer)
# - Large model with strong semantic understanding
#
# Pretraining:
# - Denoising autoencoder objective (reconstruct corrupted text)
# - Learns both understanding (encoder) and generation (decoder)
#
# Fine-tuning:
# - Fine-tuned on MNLI (Multi-Genre Natural Language Inference)
# - Learns relationships: entailment, contradiction, neutral
#
# How zero-shot works:
# - Input text is treated as a "premise"
# - Each candidate label is converted into a hypothesis sentence
#   e.g., "This text is about <LABEL>"
# - Model predicts entailment probability for each hypothesis
#
# Classification logic:
# - Label with highest entailment score is selected
# - Scores are normalized across candidate labels
#
# Output:
# - List of candidate labels with confidence scores
# - Supports multi-label and single-label classification
#
# Use cases:
# - Topic classification without labeled data
# - Rapid prototyping
# - Dynamic, user-defined categories
#
# Key insight:
# - Reframes classification as a semantic reasoning (NLI) problem
# - No classifier head trained on target labels


issue_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

labels = [
    "delivery issue",
    "payment issue",
    "app bug",
    "customer support issue",
    "refund issue",
    "login issue"
]

for text in feedbacks:
    result = issue_classifier(text, labels)
    print(text)
    print("Predicted issue:", result["labels"][0])
    print("-" * 60)

Loading weights: 100%|██████████| 515/515 [00:02<00:00, 196.86it/s, Materializing param=model.shared.weight]                                   


The delivery was late and the box was damaged
Predicted issue: delivery issue
------------------------------------------------------------
Customer support did not respond for three days
Predicted issue: customer support issue
------------------------------------------------------------
Amazing app experience, very intuitive UI
Predicted issue: payment issue
------------------------------------------------------------
Payment failed twice and money was deducted
Predicted issue: payment issue
------------------------------------------------------------
The product quality is excellent but delivery was slow
Predicted issue: delivery issue
------------------------------------------------------------
App crashes frequently after the latest update
Predicted issue: app bug
------------------------------------------------------------
Refund process was smooth and quick
Predicted issue: refund issue
------------------------------------------------------------
Terrible customer service, very di

In [2]:
# from transformers import pipeline

# Pipeline: Text Summarization
#
# Task:
# - Generate a concise summary from a longer input document
# - This is an abstractive task (model rewrites content, not just extracts sentences)
#
# Model:
# - facebook/bart-large-cnn
# - Architecture: BART (Encoder–Decoder Transformer)
# - Combines bidirectional understanding (encoder) with autoregressive generation (decoder)
#
# Pretraining:
# - Denoising autoencoder objective
# - Input text is corrupted (masking, deletion, permutation)
# - Model learns to reconstruct the original text
#
# Fine-tuning:
# - Fine-tuned on the CNN/DailyMail news summarization dataset
# - Learns to produce short, fluent summaries of long articles
#
# How summarization works:
# - Encoder reads and understands the full document
# - Decoder generates the summary token-by-token
# - Uses attention to focus on the most salient parts of the input
#
# Output:
# - A fluent natural-language summary
# - Shorter than input, preserving key information
#
# Use cases:
# - News article summarization
# - Research paper abstracts
# - Meeting notes and reports
# - Long document compression
#
# Key insight:
# - Requires encoder–decoder architecture
# - Encoder-only models (e.g., BERT) cannot perform true abstractive summarization


summarizer = pipeline(
"summarization",
model="facebook/bart-large-cnn"
)

summary = summarizer(
" ".join(negative_feedbacks),
max_length=80,
min_length=30
)

summary

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"